# 02 · Detector Agent & detector-only baseline (baseline i)

**Purpose.** Run the open-vocabulary detector over the held-out and test splits once, cache every candidate (c ≥ box_threshold), and evaluate the single-pass detector across confidence thresholds. Includes the go / no-go checkpoint from the work plan.

| | |
|---|---|
| **Proposal objective** | Objective 1 (Detector Agent), Objective 2 (baseline i) |
| **Inputs** | COCO + `results/splits.json` |
| **Outputs** | `results/detections/*.jsonl`, `results/detector/*` |
| **Runtime (T4)** | Grounding DINO-T: ≈0.5–1.5 s/img → held-out ≈10 min, test (1000) ≈20 min |

> Run cells top to bottom. Every expensive stage is **resumable**: if Colab disconnects, reconnect,
> re-run from the top and it continues where it stopped (results are cached on Drive under `results/`).

In [ ]:
# ===================== Bootstrap (identical in every notebook) =====================
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules

# Where the project lives on Google Drive. Upload the whole repo folder there (see docs/COLAB_GUIDE.md),
# or set REPO_URL and it will be cloned into DRIVE_PROJECT on first run.
DRIVE_PROJECT = "/content/drive/MyDrive/detect-verify-decide"
REPO_URL = ""  # e.g. "https://github.com/<you>/detect-verify-decide.git"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.isdir(DRIVE_PROJECT) and REPO_URL:
        subprocess.run(["git", "clone", REPO_URL, DRIVE_PROJECT], check=True)
    PROJECT_ROOT = DRIVE_PROJECT
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

assert os.path.isdir(os.path.join(PROJECT_ROOT, "src", "dvd")), f"src/dvd not found under {PROJECT_ROOT}"
os.environ["DVD_ROOT"] = PROJECT_ROOT
sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))
print("PROJECT_ROOT =", PROJECT_ROOT)

In [ ]:
# Install dependencies (Colab already has torch/torchvision with CUDA). ~1-2 min per fresh runtime.
if IN_COLAB:
    %pip install -q -r "{PROJECT_ROOT}/requirements.txt"

In [ ]:
import json, numpy as np, pandas as pd, matplotlib.pyplot as plt
from dvd.config import load_config, save_selected, results_dir
from dvd.io_utils import set_seed, save_json, load_json, JsonlCache
from dvd import paths

cfg = load_config()
set_seed(cfg["seed"])
pd.set_option("display.precision", 4)
try:
    import torch
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU only - expect very slow runs)")
except ImportError:
    print("torch not installed")
print("data_root   :", cfg["paths"]["data_root"])
print("results_root:", cfg["paths"]["results_root"])
print("tuned params:", cfg["selected"] or "(none yet)")

In [ ]:
from dvd.data.coco import prepare_coco, CocoData, load_splits
prepare_coco(cfg)          # after a runtime restart this re-extracts from the zip cached on Drive (~1-2 min)
coco = CocoData(cfg)
splits = load_splits(cfg)
HELDOUT, TEST = splits["heldout"], splits["test"]
print(f"held-out: {len(HELDOUT)} images | test: {len(TEST)} images")

In [ ]:
from dvd.agents.detector import build_detector
from dvd.pipeline import run_detection_stage
detector = build_detector(cfg, coco.cat_names, coco.name_to_cat_id)
print("backend:", cfg["detector"]["backend"], "| prompts per image:", getattr(detector, "prompts", ["(single pass)"]))

## 1. Go / no-go checkpoint (50 held-out images)
The same cache file is used, so this work is not repeated later.

In [ ]:
from dvd.metrics import DetectionEvaluator, sweep_detector_only, select_best
probe = HELDOUT[:50]
det_probe = run_detection_stage(detector, probe, paths.det_cache(cfg, "heldout"), coco.image, "detect probe")
ev = DetectionEvaluator(det_probe, coco.gts(probe), cfg["coco"]["iou_threshold"])
probe_rows = pd.DataFrame(sweep_detector_only(ev, cfg["decision"]["detector_only_grid"]))
best = probe_rows.loc[probe_rows.f1.idxmax()]
print(f"best F1 {best.f1:.3f} at tau={best.tau}: precision {best.precision:.3f}, recall {best.recall:.3f}")
print("GO" if best.precision > 0.3 and best.recall > 0.3 else "NO-GO: check detector config / label mapping before continuing")
probe_rows[["tau", "precision", "recall", "f1", "fp_per_image"]]

## 2. Full detection runs (resumable)

In [ ]:
det_h = run_detection_stage(detector, HELDOUT, paths.det_cache(cfg, "heldout"), coco.image, "detect held-out")
det_t = run_detection_stage(detector, TEST, paths.det_cache(cfg, "test"), coco.image, "detect test")
n_c = [len(r["scores"]) for r in det_t.values()]
print(f"test: {np.mean(n_c):.1f} candidates / image (max {np.max(n_c)}), c >= {cfg['detector']['box_threshold']}")

## 3. Detector-only operating curve
Threshold chosen on held-out (best F1), then applied unchanged to test.

In [ ]:
ev_h = DetectionEvaluator(det_h, coco.gts(HELDOUT), cfg["coco"]["iou_threshold"])
ev_t = DetectionEvaluator(det_t, coco.gts(TEST), cfg["coco"]["iou_threshold"])
rows_h = pd.DataFrame(sweep_detector_only(ev_h, cfg["decision"]["detector_only_grid"]))
rows_t = pd.DataFrame(sweep_detector_only(ev_t, cfg["decision"]["detector_only_grid"]))
tau_star = float(rows_h.loc[rows_h.f1.idxmax(), "tau"])
print("held-out best-F1 threshold tau* =", tau_star)
rows_t[["tau", "tp", "fp", "precision", "recall", "f1", "fp_per_image"]]

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for df, name in [(rows_h, "held-out"), (rows_t, "test")]:
    ax[0].plot(df.recall, df.precision, "-o", label=name)
    ax[1].plot(df.tau, df.fp_per_image, "-o", label=name)
ax[0].set(xlabel="recall", ylabel="precision", title="Detector-only PR"); ax[1].set(xlabel="tau", ylabel="FP / image", title="False positives vs threshold")
for a in ax: a.grid(alpha=.3); a.legend()
plt.show()

## 4. Sanity check with official COCO mAP
Candidates below `box_threshold` were discarded, so mAP is a lower bound of the detector's published value. Use it only to confirm the label mapping is right (mAP should be clearly non-trivial).

In [ ]:
from pycocotools.cocoeval import COCOeval
dets = [{"image_id": i, "category_id": c, "bbox": [b[0], b[1], b[2] - b[0], b[3] - b[1]], "score": s}
        for i, r in det_t.items() for b, s, c in zip(r["boxes"], r["scores"], r["category_ids"])]
E = COCOeval(coco.coco, coco.coco.loadRes(dets), "bbox"); E.params.imgIds = TEST
E.evaluate(); E.accumulate(); E.summarize()

## 5. Cost of the detector and save

In [ ]:
from dvd.cost import detector_cost
dcost = detector_cost(det_t); print(dcost, f"-> {1 / dcost['det_latency_s']:.2f} FPS")
out = results_dir(cfg, "detector")
rows_h.to_csv(out / "detector_only_sweep_heldout.csv", index=False)
rows_t.to_csv(out / "detector_only_sweep_test.csv", index=False)
save_json({"tau_star": tau_star, "coco_map": float(E.stats[0]), "coco_map50": float(E.stats[1]), **dcost}, out / "summary.json")
save_selected(cfg, detector_only_tau=tau_star)

## 6. Examples

In [ ]:
from dvd.viz import show_detections
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, iid in zip(axes.ravel(), TEST[:6]):
    r = det_t[iid]; keep = [k for k, s in enumerate(r["scores"]) if s >= tau_star]
    show_detections(coco.image(iid), [r["boxes"][k] for k in keep], [r["labels"][k] for k in keep],
                    [r["scores"][k] for k in keep], ax=ax, title=f"{iid} (c >= {tau_star})")
plt.tight_layout(); plt.show()

**Next:** `03_verifier_agent_crop_verification`.